# Data Loading and Preparation

## Basic Imports

In [ ]:
# Data handling
import pandas as pd

# Pathing
from pathlib import Path

# Persistence
import joblib

## Load Flagged Flow Dataset

In [ ]:
anomalies_path = Path("../data/processed/stage1_anomalies.csv")
df_anomalies = pd.read_csv(anomalies_path, low_memory=False)
df_features = df_anomalies.copy() # Working copy for second-stage feature engineering
df_features.head()

# Time-Based Features

## Start Hour of Day

An `Shourofday` feature is created by extracting the `hour` attribute from the `Stime` column.
- Highlights attacks that occur outside normal working hours.

In [ ]:
Sdt = pd.to_datetime(df_features["Stime"], unit="s")
df_features["Shourofday"] = Sdt.dt.hour

## Last Hour of Day

An `Lhourofday` feature is created by extracting the `hour` attribute from the `Ltime` column.
- Highlights attacks that occur outside normal working hours.

In [ ]:
Ldt = pd.to_datetime(df_features["Ltime"], unit="s")
df_features["Lhourofday"] = Ldt.dt.hour

# TTL Features

## TTL Difference Bin

A binary `ttl_diff_bin` feature is created from `ttl_diff` values.
- 1 indicates values above 250 (TP-like).
- 0 indicates values 250 or below (FP-like).
- Provides a clear signal for the model to distinguish true positives from false positives.

In [ ]:
df_features["ttl_diff_bin"] = (df_features["ttl_diff"] > 250).astype(int)

## TTL Difference Low

A binary `ttl_diff_low` feature is created from `ttl_diff` values.
- 1 indicates low values between 200-250 (likely true positives).
- 0 indicates values outside that range (FP-like).
- Provides a second categorical cue for TPs just below the main cluster.

In [ ]:
# tp_low_threshold = 200
# df_features["ttl_diff_low"] = ((df_features["ttl_diff"] > tp_low_threshold) & (df_features["ttl_diff"] <= 250)).astype(int)

This feature was ultimately redundant: the numeric `ttl_diff` already captured the necessary signal, and XGBoost did not leverage the binary cue.

## STTL X TTL Difference

A `sttl_x_ttl_diff` feature is created by taking the product of the `sttl` and `ttl_diff` columns.
- Highlights unusual combinations that may be indicative of anomalies.

In [ ]:
df_features["sttl_x_ttl_diff"] = df_features["sttl"]*df_features["ttl_diff"]

## STTL X CT_State_TTL

A `sttl_x_ct_state_ttl` feature is created by taking the product of the `sttl` and `ct_state_ttl` columns.
- Captures how unusual a packet's TTL is given its state frequency.
- Signals rare of suspicious TTL-state combinations.

In [ ]:
df_features["sttl_x_ct_state_ttl"] = df_features["sttl"]*df_features["ct_state_ttl"]

# Categorical Features

## Replace Infrequent Categories With "Other"

In [ ]:
top_k = 10 # Keep 10 most frequent categories
for col in ["state", "service", "proto", "dsport"]:
    counts = df_features[col].value_counts()
    top_categories = counts.nlargest(top_k).index
    df_features[col] = df_features[col].where(df_features[col].isin(top_categories), other="other")

# Export Final Feature Set

## Select Features

### Define List

In [ ]:
stage2_selected_features = [
    "ct_state_ttl",
    "Sintpkt",
    "sttl",
    "trans_depth",
    "ttl_diff",
    "ttl_diff_bin",
    "dloss",
    "res_bdy_len",
    "Dintpkt",
    "mean_pkt_sz_ratio",
    "ct_dst_src_ltm",
    "dur",
    "anomaly_score",
    "Shourofday",
    "sloss",
    "sttl_x_ct_state_ttl",
    "sttl_x_ttl_diff",
    "state",
    "service",
    "proto",
    "dsport"
]

### Save Features to Disk

In [ ]:
stage2_selected_features_path = Path("../data/processed/stage2_selected_features.pkl")
joblib.dump(stage2_selected_features, stage2_selected_features_path)
print(f"Selected features saved to {stage2_selected_features_path}.")

## Save Subset to CSV

In [ ]:
# Save df_features to CSV for modelling
features_path = Path("../data/processed/stage2_features.csv")
df_features[stage2_selected_features + ["Label"]].to_csv(features_path, index=False)